<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h1>Notebook Modelisation - ALS (Spark MLlib)</h1>
<p>Entraine un modele ALS sur les splits temporels prepares dans l'EDA<br>
selectionne les hyperparametres sur validation et evalue sur test.</p>
</div>

In [2]:
from pathlib import Path

import pandas as pd
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import functions as F

import src.utils as utils

In [ ]:
# Configuration experimentale (meme naming que l'EDA)

# "processed_small" ou "processed_big"
DATA_SOURCE = "processed_small"  
RANDOM_SEED = 42
TOP_N = 10

if DATA_SOURCE not in {"processed_small", "processed_big"}:
    raise ValueError(
        f"DATA_SOURCE invalide pour ce notebook: {DATA_SOURCE}. Utiliser processed_small ou processed_big."
    )

DATA_SIZE = DATA_SOURCE.replace("processed_", "")

# Grille simple pour debuter
GRID_RANK = [20, 40]
GRID_REG_PARAM = [0.05, 0.1]
GRID_MAX_ITER = [10, 15]

In [11]:
spark = utils.create_spark_session()
project_root = utils.get_project_root()

# Meme procede que l'EDA: on resout les chemins a partir de DATA_SOURCE
dataset_format, path_ratings, path_movies = utils.resolve_data_source_paths(
    DATA_SOURCE, project_root=project_root
)
if dataset_format != "parquet":
    raise ValueError(
        f"Ce notebook ALS attend une source processed_* en parquet, recu: {DATA_SOURCE} ({dataset_format})"
    )

processed_root = project_root / "data" / "processed" / DATA_SIZE
split_root = processed_root / "splits_temporal"

required_paths = [
    path_ratings,
    path_movies,
    split_root / "train",
    split_root / "validation",
    split_root / "test",
]

missing = [p for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Artefacts manquants: {missing}")

df_movies_clean = spark.read.parquet(path_movies.as_posix())
df_train = spark.read.parquet((split_root / "train").as_posix())
df_val = spark.read.parquet((split_root / "validation").as_posix())
df_test = spark.read.parquet((split_root / "test").as_posix())

for name in ["df_train", "df_val", "df_test"]:
    df = globals()[name]
    globals()[name] = df.select(
        F.col("userId").cast("int").alias("userId"),
        F.col("movieId").cast("int").alias("movieId"),
        F.col("rating").cast("float").alias("rating"),
        F.col("timestamp").cast("long").alias("timestamp"),
    )

print(f"Artefacts charges avec succes depuis DATA_SOURCE={DATA_SOURCE}.")

Artefacts charges avec succes depuis DATA_SOURCE=processed_small.


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Chargement des artefacts</h3>
Si le message affiche <b>Artefacts charges avec succes</b>, la base experimentale est prete.

Verification implicite:
- Les fichiers nettoyes et les splits temporels existent bien.
- Les colonnes userId, movieId, rating, timestamp ont ete castees au bon type pour ALS.

En cas d'erreur, il faut relancer le notebook EDA pour regenerer les artefacts.
</div>

In [5]:
print("--- Tailles des splits ---")
print(f"Train: {df_train.count()}")
print(f"Validation: {df_val.count()}")
print(f"Test: {df_test.count()}")

print("\n--- Cardinalites train ---")
print(f"Users train: {df_train.select('userId').distinct().count()}")
print(f"Items train: {df_train.select('movieId').distinct().count()}")

--- Tailles des splits ---
Train: 80669
Validation: 10084
Test: 10083

--- Cardinalites train ---
Users train: 522
Items train: 7867


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Taille des splits</h3>
Cette sortie confirme la proportion train/validation/test et la cardinalite users/items en train.

Comment interpreter:
- Train doit etre majoritaire pour apprendre correctement.
- Validation sert au choix des hyperparametres.
- Test reste strictement reserve a l'evaluation finale.

Si les cardinalites sont tres faibles, reduire la complexite du modele (rank, iterations).
</div>

In [6]:
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

results = []
best_rmse = float("inf")
best_params = None

for rank in GRID_RANK:
    for reg_param in GRID_REG_PARAM:
        for max_iter in GRID_MAX_ITER:
            als = ALS(
                userCol="userId",
                itemCol="movieId",
                ratingCol="rating",
                rank=rank,
                regParam=reg_param,
                maxIter=max_iter,
                coldStartStrategy="drop",
                nonnegative=True,
                seed=RANDOM_SEED,
            )

            model = als.fit(df_train)
            pred_val = model.transform(df_val).dropna(subset=["prediction"])
            rmse_val = evaluator.evaluate(pred_val)

            results.append({
                "rank": rank,
                "regParam": reg_param,
                "maxIter": max_iter,
                "rmse_val": rmse_val,
            })

            if rmse_val < best_rmse:
                best_rmse = rmse_val
                best_params = (rank, reg_param, max_iter)

df_grid = pd.DataFrame(results).sort_values("rmse_val")
display(df_grid)
print(f"Meilleurs params (val): rank={best_params[0]}, regParam={best_params[1]}, maxIter={best_params[2]}")
print(f"RMSE validation: {best_rmse:.4f}")

26/03/20 17:24:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


,rank,regParam,maxIter,rmse_val
6,40,0.10,10,0.940857
7,40,0.10,15,0.941185
3,20,0.10,15,0.944285
2,20,0.10,10,0.945507
1,20,0.05,15,0.992561
5,40,0.05,15,0.998106
0,20,0.05,10,1.008244
4,40,0.05,10,1.020663


Meilleurs params (val): rank=40, regParam=0.1, maxIter=10
RMSE validation: 0.9409


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Tuning ALS sur validation</h3>
Le tableau affiche chaque combinaison (rank, regParam, maxIter) et son RMSE validation.

Regle de lecture:
- Plus le RMSE est bas, meilleure est la prediction de note sur validation.
- La ligne retenue devient la configuration du modele final.

Attention:
- Un bon score validation ne garantit pas toujours un bon score test.
- Le test final sert a verifier la generalisation.
</div>

In [7]:
# Re-entraine sur train + validation puis evalue sur test
df_train_val = df_train.unionByName(df_val)

best_rank, best_reg_param, best_max_iter = best_params
als_final = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=best_rank,
    regParam=best_reg_param,
    maxIter=best_max_iter,
    coldStartStrategy="drop",
    nonnegative=True,
    seed=RANDOM_SEED,
)

model_final = als_final.fit(df_train_val)
pred_test = model_final.transform(df_test).dropna(subset=["prediction"])
rmse_test = evaluator.evaluate(pred_test)

print(f"RMSE test (modele final): {rmse_test:.4f}")

RMSE test (modele final): 0.9020


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - RMSE test</h3>
Ce score est la mesure principale de performance hors echantillon.

Interpretation:
- Plus le RMSE test est bas, plus la prediction des notes est precise.
- Si RMSE test est nettement pire que validation, il peut y avoir surapprentissage.

Ce score servira de reference pour comparer ensuite contenu et KNN.
</div>

In [8]:
# Recommandations Top-N pour quelques utilisateurs
sample_users = [r.userId for r in df_train.select("userId").distinct().limit(5).collect()]
df_users = spark.createDataFrame([(u,) for u in sample_users], ["userId"])

df_reco = model_final.recommendForUserSubset(df_users, TOP_N)
df_reco_flat = (
    df_reco
    .withColumn("rec", F.explode("recommendations"))
    .select(
        "userId",
        F.col("rec.movieId").alias("movieId"),
        F.col("rec.rating").alias("score"),
    )
    .join(df_movies_clean.select("movieId", "title"), on="movieId", how="left")
    .orderBy("userId", F.desc("score"))
)

df_reco_flat.show(50, truncate=False)

+-------+------+---------+---------------------------------------------------------------------------+
|movieId|userId|score    |title                                                                      |
+-------+------+---------+---------------------------------------------------------------------------+
|59814  |243   |5.3983574|Ex Drummer (2007)                                                          |
|74946  |243   |5.3394585|She's Out of My League (2010)                                              |
|86377  |243   |5.278813 |Louis C.K.: Shameless (2007)                                               |
|138966 |243   |5.2779603|Nasu: Summer in Andalusia (2003)                                           |
|134796 |243   |5.2779603|Bitter Lake (2015)                                                         |
|117531 |243   |5.2779603|Watermark (2014)                                                           |
|86237  |243   |5.2779603|Connections (1978)                             

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Top-N recommandations</h3>
Le tableau liste les films recommandes pour quelques utilisateurs de test.

Interpretation des colonnes:
- userId: utilisateur cible.
- movieId / title: film recommande.
- score: score predit par ALS (plus eleve = recommandation plus forte).

Controle qualite simple:
- Verifier la diversite des titres proposes.
- Verifier que les recommendations semblent plausibles pour chaque utilisateur.
</div>

In [9]:
spark.stop()
print("Session Spark arretee.")

Session Spark arretee.


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Bilan ALS</h3>
Le pipeline ALS est complet:
- chargement des artefacts
- tuning sur validation
- evaluation sur test
- generation des recommandations top-N

Prochaine etape recommandee: ajouter Precision@K, Recall@K et coverage pour comparer ALS, contenu et KNN sur un protocole unique.
</div>